# 04 — Qui consomme quoi ? Profils démographiques et de prescription

**Objectif** : caractériser les patients et les circuits de prescription
derrière chaque produit — sexe, âge et son évolution dans le temps,
concentration de la prescription, répartition régionale.


## 0. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

COULEURS = {'Trulicity': '#0072B2', 'Verzenio': '#D55E00',
            'Jardiance': '#009E73', 'Cymbalta': '#CC79A7'}
PRODUITS = ['Trulicity', 'Verzenio', 'Jardiance', 'Cymbalta']

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_FILE   = PROJECT_DIR / 'data' / 'eli_lilly.csv'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)
print(f'{df.shape[0]:,} lignes chargées')

---
## 1. Qui est traité ? Sexe et âge par produit

In [ ]:
sexe = (df.groupby(['nom_lilly', 'sexe_label'])['boites'].sum()
        .unstack('sexe_label').fillna(0))
sexe_pct = sexe.div(sexe.sum(axis=1), axis=0) * 100
sexe_pct = sexe_pct.reindex(PRODUITS)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(PRODUITS, sexe_pct['Femme'], color='#CC79A7', label='Femme')
ax.barh(PRODUITS, sexe_pct['Homme'], left=sexe_pct['Femme'], color='#0072B2', label='Homme')
for i, p in enumerate(PRODUITS):
    ax.text(sexe_pct.loc[p, 'Femme'] / 2, i, f"{sexe_pct.loc[p, 'Femme']:.0f} %",
            va='center', ha='center', color='white', fontsize=9, fontweight='bold')
    ax.text(sexe_pct.loc[p, 'Femme'] + sexe_pct.loc[p, 'Homme'] / 2, i,
            f"{sexe_pct.loc[p, 'Homme']:.0f} %", va='center', ha='center',
            color='white', fontsize=9, fontweight='bold')
ax.set_xlabel('Part des boîtes remboursées (%)')
ax.set_title('Répartition par sexe, par produit')
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1.15), frameon=False)
plt.tight_layout()
plt.show()

sexe_pct.round(1)

**Lecture.** Verzenio (cancer du sein) est traité à **99 % par des femmes** —
cohérent avec son indication clinique. Cymbalta est à 75 % féminin, en ligne
avec la prévalence plus élevée de la dépression diagnostiquée chez les femmes.
Trulicity et Jardiance, traitements du diabète de type 2, sont quasi équilibrés
entre les sexes.

In [ ]:
age_order = ['<20 ans', '20-59 ans', '60+ ans']
age = (df[df['age'] != 99].groupby(['nom_lilly', 'age_label'])['boites'].sum()
       .unstack('age_label').fillna(0).reindex(columns=age_order))
age_pct = age.div(age.sum(axis=1), axis=0) * 100
age_pct = age_pct.reindex(PRODUITS)

fig, ax = plt.subplots(figsize=(9, 5.5))
gauche = np.zeros(len(PRODUITS))
couleurs_age = ['#E69F00', '#56B4E9', '#0072B2']
for tranche, c in zip(age_order, couleurs_age):
    vals = age_pct[tranche].values
    ax.barh(PRODUITS, vals, left=gauche, color=c, label=tranche)
    gauche += vals
ax.set_xlabel('Part des boîtes remboursées (%)')
ax.set_title('Répartition par tranche d\'âge, par produit')
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.15), frameon=False)
plt.tight_layout()
plt.show()

age_pct.round(1)

**Lecture.** Jardiance (84 % chez les 60 ans et plus) et Trulicity (68 %)
reflètent le profil classique du diabète de type 2, plus fréquent avec l'âge.
Verzenio penche aussi vers les 60 ans et plus (cancer du sein plus fréquent
après la ménopause), tandis que Cymbalta est le plus équilibré des quatre.

---
## 2. Trulicity vieillit-il ? Déformation de la structure d'âge dans le temps

Le battage médiatique autour des GLP-1 (Ozempic, Wegovy...) suggère un usage
croissant chez des patients plus jeunes, hors indication stricte du diabète —
un phénomène documenté aux États-Unis. **Qu'en est-il de Trulicity, en France,
dans les données de remboursement ?**

In [ ]:
tru = df[(df['nom_lilly'] == 'Trulicity') & (df['age'] != 99)]
evo_age = (tru.groupby(['annee', 'age_label'])['boites'].sum()
           .unstack('age_label').fillna(0).reindex(columns=age_order))
evo_age_pct = evo_age.div(evo_age.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(evo_age_pct.index, [evo_age_pct[t] for t in age_order],
             colors=couleurs_age, labels=age_order, alpha=0.9)
ax.set_xlabel('Année')
ax.set_ylabel('Part des boîtes (%)')
ax.set_title('Trulicity — évolution de la structure d\'âge, 2016–2025')
ax.set_ylim(0, 100)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout()
plt.show()

print(f"Part des 60+ ans : {evo_age_pct['60+ ans'].iloc[0]:.1f} % en {evo_age_pct.index[0]} "
      f"→ {evo_age_pct['60+ ans'].iloc[-1]:.1f} % en {evo_age_pct.index[-1]}")
print(f"Part des 20-59 ans : {evo_age_pct['20-59 ans'].iloc[0]:.1f} % → {evo_age_pct['20-59 ans'].iloc[-1]:.1f} %")

**Lecture — un résultat qui va à l'encontre de l'intuition médiatique.** La part
des 60 ans et plus **augmente** continûment (55 % → 73 %), tandis que celle des
20-59 ans recule. Rien, dans ces données françaises, ne suggère une diffusion
croissante vers des patients plus jeunes. C'est cohérent avec le cadre de
remboursement français : contrairement aux usages hors indication rapportés
aux États-Unis, Trulicity n'est remboursé en France que pour le diabète de
type 2 — une maladie dont la prévalence augmente avec l'âge. Le vieillissement
progressif de la base de patients traités est donc la signature attendue d'un
traitement chronique de plus en plus mature, pas un signal de mésusage.

---
## 3. Qui prescrit ? Diversité et concentration des circuits de prescription

Open Medic fournit un code de spécialité du prescripteur (`psp_spe`), mais sa
table de correspondance officielle (libellés en clair) n'est pas incluse dans
les données publiques. On mesure donc la **concentration** de la prescription
avec un indice Herfindahl-Hirschman (HHI), sans prétendre nommer les
spécialités — c'est un indicateur robuste même sans connaître le libellé exact
de chaque code.

$$HHI = 10\,000 \times \sum_i s_i^2 \quad \text{où } s_i \text{ = part du code } i \text{ dans les boîtes}$$

In [ ]:
concentration = []
for p in PRODUITS:
    parts = df[df['nom_lilly'] == p].groupby('psp_spe')['boites'].sum()
    parts = parts / parts.sum()
    hhi = (parts ** 2).sum() * 10_000
    n_codes = (parts > 0).sum()
    n_codes_90 = (parts.sort_values(ascending=False).cumsum() <= 0.9).sum() + 1
    concentration.append({'produit': p, 'HHI': hhi, 'n_codes_distincts': n_codes,
                          'n_codes_pour_90pct': n_codes_90})

conc_df = pd.DataFrame(concentration).set_index('produit').reindex(PRODUITS)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(PRODUITS, conc_df['HHI'], color=[COULEURS[p] for p in PRODUITS])
axes[0].axhline(2500, color='#999999', linestyle='--', linewidth=1)
axes[0].text(0, 2550, 'seuil "hautement concentré" (HHI > 2500)', fontsize=7.5, color='#666')
axes[0].set_ylabel('HHI (0–10 000)')
axes[0].set_title('Concentration de la prescription (HHI)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(PRODUITS, conc_df['n_codes_distincts'], color=[COULEURS[p] for p in PRODUITS])
axes[1].set_ylabel('Nombre de codes prescripteurs distincts')
axes[1].set_title('Diversité des circuits de prescription')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()
conc_df

**Lecture.** Le HHI seul est trompeur ici : les quatre produits ont un indice
élevé et similaire (un code domine toujours à 65-80 %). La différence
intéressante est ailleurs : Verzenio circule à travers seulement **6 codes**
prescripteurs distincts sur toute la période, contre **23** pour Trulicity et
Jardiance. Cela suggère un circuit de soin beaucoup plus étroit et
institutionnel pour l'oncologie — cohérent avec une prescription hospitalière
encadrée — face à des traitements chroniques du diabète qui irriguent un
éventail bien plus large de contextes de prescription.

---
## 4. Répartition régionale

**Note de qualité des données.** Le dictionnaire de codes régionaux hérité des
notebooks précédents associait le code `5` à « Mayotte ». Une vérification
croisée a montré que ce code porte à lui seul **1,33 million de boîtes** — un
volume comparable aux plus grandes régions métropolitaines — alors que les
codes `1` à `4` (Guadeloupe, Martinique, Guyane, La Réunion) sont
**totalement absents** des données. Le code officiel INSEE de Mayotte est en
réalité `06`, absent lui aussi. Le code `5` regroupe donc très probablement
l'ensemble des **DOM/TOM agrégés**, pas la seule Mayotte — il est renommé en
conséquence ci-dessous. *(Si ce point est important pour votre soutenance,
vérifiez le fichier officiel « Descriptif des variables Open Medic » d'AMELI.)*

In [ ]:
REG_LABELS_CORRIGE = {
    5: 'DOM/TOM (agrégés, code 5)',
    11: 'Île-de-France', 24: 'Centre-Val de Loire', 27: 'Bourgogne-Franche-Comté',
    28: 'Normandie', 32: 'Hauts-de-France', 44: 'Grand Est',
    52: 'Pays de la Loire', 53: 'Bretagne', 75: 'Nouvelle-Aquitaine',
    76: 'Occitanie', 84: 'Auvergne-Rhône-Alpes', 93: 'PACA', 94: 'Corse',
}
df['reg_corrige'] = df['ben_reg'].map(REG_LABELS_CORRIGE)

df_reg = df[df['ben_reg'] != 99]
croise = (df_reg.groupby(['nom_lilly', 'reg_corrige'])['boites'].sum()
          .unstack('nom_lilly').fillna(0))
croise_pct = croise.div(croise.sum(axis=0), axis=1) * 100
ordre_regions = croise_pct.sum(axis=1).sort_values(ascending=False).index
croise_pct = croise_pct.loc[ordre_regions, PRODUITS]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(croise_pct.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(PRODUITS)))
ax.set_xticklabels(PRODUITS, rotation=20, ha='right')
ax.set_yticks(range(len(ordre_regions)))
ax.set_yticklabels(ordre_regions, fontsize=8.5)
ax.grid(False)

seuil = croise_pct.values.max() * 0.55
for i in range(croise_pct.shape[0]):
    for j in range(croise_pct.shape[1]):
        v = croise_pct.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7.5,
                color='white' if v > seuil else '#333333')

cbar = fig.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label('Part du produit dans la région (%)')
ax.set_title('Répartition régionale des boîtes, par produit (%)')
plt.tight_layout()
plt.show()

**Lecture.** La structure régionale est globalement proche entre produits — les
grandes régions (Île-de-France, Auvergne-Rhône-Alpes) captent naturellement
plus de volume, proportionnellement à leur population. Aucune région ne se
distingue par une spécialisation marquée sur un produit particulier, ce qui est
attendu : ni la prévalence du diabète, ni celle du cancer du sein ou de la
dépression, ne varient au point de créer une géographie de consommation très
différenciée en France métropolitaine.

---
## 5. Synthèse et export

**Ce qu'il faut retenir :**
- Les profils démographiques suivent fidèlement l'épidémiologie des
  indications cliniques (Verzenio quasi exclusivement féminin, Jardiance
  concentré chez les 60 ans et plus).
- Trulicity **vieillit** avec le temps en France — aucun signe d'une diffusion
  vers des usages plus jeunes comme celle documentée pour les GLP-1 aux
  États-Unis, ce qui s'explique par un remboursement strictement limité à
  l'indication diabète.
- Verzenio circule dans un cercle de prescripteurs bien plus étroit que les
  traitements chroniques du diabète — cohérent avec un circuit hospitalier
  spécialisé pour l'oncologie.
- Une erreur de labellisation régionale héritée d'un projet précédent a été
  identifiée et corrigée : le code 5 regroupe des DOM/TOM, ce n'est pas
  Mayotte seule.

In [ ]:
export = sexe_pct.join(age_pct, lsuffix='_sexe', rsuffix='_age').join(conc_df)
export_path = OUTPUTS_DIR / 'profils_patients.csv'
export.to_csv(export_path, encoding='utf-8')
print(f'Exporté : {export_path}')
export.round(1)